# Linear Discriminant Analysis (LDA)

Linear Discriminant Analysis is a classification method that projects data onto a lower-dimensional space to maximize class separability. It's a supervised dimensionality reduction technique that also serves as a classifier.

## Key Concepts:

- **Linear Decision Boundaries**: Creates linear decision boundaries between classes
- **Dimensionality Reduction**: Projects data onto a line (or hyperplane) that maximizes class separation
- **Assumptions**: Assumes classes have identical covariance matrices and features are normally distributed
- **Bayes Classifier**: Can be viewed as a special case of Bayes classifier with shared covariance

## When to Use:

- When classes are well-separated and linearly separable
- When you have limited training data
- When computational efficiency is important
- When you want both classification and dimensionality reduction

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, make_classification
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Dataset 1: Iris Dataset

We'll start with the classic Iris dataset.

In [ ]:
# Load Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

print(f"Feature shape: {X.shape}")
print(f"Classes: {iris.target_names}")
print(f"Features: {iris.feature_names}")
print(f"\nClass distribution: {np.bincount(y)}")

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Scale features (important for LDA)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training samples: {X_train_scaled.shape[0]}")
print(f"Test samples: {X_test_scaled.shape[0]}")

## Train LDA Model

In [ ]:
# Initialize and train LDA
lda = LinearDiscriminantAnalysis()
lda.fit(X_train_scaled, y_train)

# Make predictions
y_pred = lda.predict(X_test_scaled)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Display explained variance ratio
print(f"\nExplained variance ratio: {lda.explained_variance_ratio_}")

## Model Evaluation

In [ ]:
# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - LDA on Iris')
plt.tight_layout()
plt.show()

## Dimensionality Reduction with LDA

LDA can reduce dimensions while preserving class separation. For 3 classes, we can reduce to at most 2 dimensions.

In [ ]:
# Transform data to 2D using LDA
lda_2d = LinearDiscriminantAnalysis(n_components=2)
X_train_2d = lda_2d.fit_transform(X_train_scaled, y_train)
X_test_2d = lda_2d.transform(X_test_scaled)

print(f"Original dimensions: {X_train_scaled.shape[1]}")
print(f"Reduced dimensions: {X_train_2d.shape[1]}")
print(f"Explained variance ratio: {lda_2d.explained_variance_ratio_}")

In [ ]:
# Visualize the 2D projection
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_train_2d[:, 0], X_train_2d[:, 1], c=y_train, cmap='viridis', alpha=0.7)
plt.xlabel('LD1')
plt.ylabel('LD2')
plt.title('LDA: 2D Projection of Iris Data')
plt.colorbar(scatter, label='Class')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Decision Boundary Visualization

In [ ]:
# Create mesh grid for decision boundary
h = 0.02
x_min, x_max = X_train_2d[:, 0].min() - 1, X_train_2d[:, 0].max() + 1
y_min, y_max = X_train_2d[:, 1].min() - 1, X_train_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Train LDA on 2D data
lda_2d_classifier = LinearDiscriminantAnalysis()
lda_2d_classifier.fit(X_train_2d, y_train)

# Predict on mesh grid
Z = lda_2d_classifier.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot decision boundary
plt.figure(figsize=(12, 8))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
scatter = plt.scatter(X_train_2d[:, 0], X_train_2d[:, 1], c=y_train, cmap='viridis', edgecolors='k')
plt.xlabel('LD1')
plt.ylabel('LD2')
plt.title('LDA Decision Boundary on 2D Projection')
plt.colorbar(scatter, label='Class')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Dataset 2: Synthetic Classification Dataset

Let's test LDA on a synthetic dataset with more features.

In [ ]:
# Create synthetic dataset
X_syn, y_syn = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    n_classes=3,
    n_clusters_per_class=1,
    random_state=42
)

print(f"Synthetic dataset shape: {X_syn.shape}")
print(f"Class distribution: {np.bincount(y_syn)}")

In [ ]:
# Split and scale
X_train_syn, X_test_syn, y_train_syn, y_test_syn = train_test_split(
    X_syn, y_syn, test_size=0.3, random_state=42, stratify=y_syn
)

scaler_syn = StandardScaler()
X_train_syn_scaled = scaler_syn.fit_transform(X_train_syn)
X_test_syn_scaled = scaler_syn.transform(X_test_syn)

# Train LDA
lda_syn = LinearDiscriminantAnalysis()
lda_syn.fit(X_train_syn_scaled, y_train_syn)

# Predict and evaluate
y_pred_syn = lda_syn.predict(X_test_syn_scaled)
accuracy_syn = accuracy_score(y_test_syn, y_pred_syn)

print(f"Accuracy on synthetic dataset: {accuracy_syn:.4f}")
print(f"Explained variance ratio: {lda_syn.explained_variance_ratio_}")

## Cross-Validation

In [ ]:
# Perform cross-validation
cv_scores = cross_val_score(lda, X_train_scaled, y_train, cv=5)

print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

## Comparison with Different Solvers

LDA supports different solvers: 'svd', 'lsqr', and 'eigen'.

In [ ]:
# Test different solvers
solvers = ['svd', 'lsqr', 'eigen']

for solver in solvers:
    lda_solver = LinearDiscriminantAnalysis(solver=solver)
    lda_solver.fit(X_train_scaled, y_train)
    y_pred_solver = lda_solver.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred_solver)
    print(f"Solver: {solver:6s}, Accuracy: {acc:.4f}")

## Predict on New Data

In [ ]:
# Function to predict new samples
def predict_iris(sample):
    sample_scaled = scaler.transform([sample])
    prediction = lda.predict(sample_scaled)[0]
    probabilities = lda.predict_proba(sample_scaled)[0]
    
    class_name = iris.target_names[prediction]
    return class_name, probabilities

# Test with new samples
new_samples = [
    [5.1, 3.5, 1.4, 0.2],  # Likely setosa
    [6.3, 2.8, 5.1, 1.5],  # Likely versicolor
    [6.4, 3.2, 5.3, 2.3]   # Likely virginica
]

for sample in new_samples:
    class_name, probs = predict_iris(sample)
    print(f"\nSample: {sample}")
    print(f"Predicted: {class_name}")
    print("Probabilities:")
    for name, prob in zip(iris.target_names, probs):
        print(f"  {name}: {prob:.4f}")

## Summary

### Key Takeaways:

1. **Linear Decision Boundaries**: LDA creates linear decision boundaries between classes
2. **Dimensionality Reduction**: Can reduce dimensions while maximizing class separation
3. **Assumptions**: Assumes equal covariance matrices and normal distribution of features
4. **Efficient**: Computationally efficient, works well with small to medium datasets

### Advantages:
- Simple and interpretable
- Provides dimensionality reduction
- Works well when classes are linearly separable
- Fast training and prediction
- Handles multi-class classification naturally

### Limitations:
- Assumes linear separability
- Assumes equal covariance matrices across classes
- Sensitive to outliers
- May underperform with non-linear decision boundaries
- Requires more samples than features for stability